In [1]:
import osmnx as ox
import pandas as pd
from geopy.distance import geodesic

In [2]:
place = "Bengaluru, Karnataka, India"

metro_tags = {
    "railway": "station"
}

metro = ox.features_from_place(place, metro_tags)

print(metro.head())
print()
print("Total stations:", len(metro))

                                    geometry internet_access  \
element id                                                     
node    246397026    POINT (77.598 12.99369)            wlan   
        253454039  POINT (77.61835 13.00115)              no   
        308062102  POINT (77.48376 12.91766)            wlan   
        308062851  POINT (77.52121 12.94133)              no   
        308073351  POINT (77.50564 13.07468)              no   

                                   name                 name:kn  \
element id                                                        
node    246397026  Bangalore Cantonment  ಬೆಂಗಳೂರು (ಕಂಟೊನ್ಮೆಂಟ್)   
        253454039        Bengaluru East          ಬೆಂಗಳೂರು ಪೂರ್ವ   
        308062102               Kengeri                 ಕೆಂಗೇರಿ   
        308062851          Nayandahalli              ನಾಯಂಡಹಳ್ಳಿ   
        308073351       Chikka Banavara            ಚಿಕ್ಕ ಬಾಣವಾರ   

                                 name:ml network operator  railway   ref  \
eleme

In [3]:
metro = metro.reset_index()

metro = metro[["name", "geometry"]]

metro["geometry"] = metro["geometry"].representative_point()

metro["latitude"] = metro.geometry.y
metro["longitude"] = metro.geometry.x

metro = metro[["name", "latitude", "longitude"]]

print(metro.head())

                   name   latitude  longitude
0  Bangalore Cantonment  12.993687  77.598000
1        Bengaluru East  13.001146  77.618345
2               Kengeri  12.917657  77.483757
3          Nayandahalli  12.941325  77.521212
4       Chikka Banavara  13.074679  77.505637


In [4]:
metro.to_csv(
    '../data/processed/bangalore_metro_stations.csv',
    index=False
)

print("Metro station dataset saved!")

Metro station dataset saved!


In [5]:
def nearest_distance(lat, lon, locations_df):
    distances = locations_df.apply(
        lambda row: geodesic(
            (lat, lon),
            (row['latitude'], row['longitude'])
        ).km,
        axis=1
    )
    return distances.min()

In [6]:
sample_lat = 12.9716
sample_lon = 77.5946

metro_distance = nearest_distance(sample_lat, sample_lon, metro)

print(f"Nearest metro station: {metro_distance:.2f} km")

Nearest metro station: 0.85 km


In [7]:
import osmnx as ox
import pandas as pd
import os

# ============================================================
# CITIES TO PROCESS
# ============================================================

cities = {
    "mumbai": "Mumbai, Maharashtra, India",
    "pune": "Pune, Maharashtra, India",
    "delhi": "Delhi, India",
    "nagpur": "Nagpur, Maharashtra, India"
}

SAVE_DIR = "../data/processed"

# ============================================================
# FUNCTION TO EXTRACT COORDINATES SAFELY
# ============================================================

def prepare_locations(gdf, category):
    gdf = gdf.reset_index()

    # Convert polygons/lines into representative points
    gdf["geometry"] = gdf["geometry"].representative_point()

    gdf["latitude"] = gdf.geometry.y
    gdf["longitude"] = gdf.geometry.x

    if "name" not in gdf.columns:
        gdf["name"] = None

    result = gdf[
        ["name", "latitude", "longitude"]
    ].copy()

    result["category"] = category

    return result


# ============================================================
# PROCESS EACH CITY
# ============================================================

for city_key, place in cities.items():

    print("\n" + "=" * 70)
    print(f"PROCESSING: {place}")
    print("=" * 70)

    # --------------------------------------------------------
    # HOSPITALS
    # --------------------------------------------------------

    print("\nDownloading hospitals...")

    hospitals_raw = ox.features_from_place(
        place,
        {"amenity": "hospital"}
    )

    hospitals = prepare_locations(
        hospitals_raw,
        "hospital"
    )

    hospitals.to_csv(
        f"{SAVE_DIR}/{city_key}_hospitals.csv",
        index=False
    )

    print("Hospitals:", len(hospitals))


    # --------------------------------------------------------
    # SCHOOLS
    # --------------------------------------------------------

    print("Downloading schools...")

    schools_raw = ox.features_from_place(
        place,
        {"amenity": "school"}
    )

    schools = prepare_locations(
        schools_raw,
        "school"
    )

    schools.to_csv(
        f"{SAVE_DIR}/{city_key}_schools.csv",
        index=False
    )

    print("Schools:", len(schools))


    # --------------------------------------------------------
    # MALLS
    # --------------------------------------------------------

    print("Downloading malls...")

    malls_raw = ox.features_from_place(
        place,
        {"shop": "mall"}
    )

    malls = prepare_locations(
        malls_raw,
        "mall"
    )

    malls.to_csv(
        f"{SAVE_DIR}/{city_key}_malls.csv",
        index=False
    )

    print("Malls:", len(malls))


    # --------------------------------------------------------
    # METRO / RAIL STATIONS
    # --------------------------------------------------------

    print("Downloading metro/rail stations...")

    metro_raw = ox.features_from_place(
        place,
        {"railway": "station"}
    )

    metro = prepare_locations(
        metro_raw,
        "metro"
    )

    metro.to_csv(
        f"{SAVE_DIR}/{city_key}_metro_stations.csv",
        index=False
    )

    print("Metro/Rail stations:", len(metro))


    print(f"\n✓ {place} completed")


print("\n" + "=" * 70)
print("ALL FOUR CITIES COMPLETED")
print("=" * 70)


PROCESSING: Mumbai, Maharashtra, India



TypeError: Nominatim did not geocode query 'Mumbai, Maharashtra, India' to a geometry of type (Multi)Polygon.

In [ ]:
import osmnx as ox

place = "Greater Mumbai, Maharashtra, India"

gdf = ox.geocode_to_gdf(place)

print(gdf[['display_name']])
print(gdf.geometry.geom_type)

                                     display_name
0  Mumbai Metropolitan Region, Maharashtra, India
0    Polygon
dtype: str


In [8]:
import osmnx as ox

ox.settings.requests_timeout = 300

print("OSMnx ready")
print("Timeout:", ox.settings.requests_timeout)

OSMnx ready
Timeout: 300


In [9]:
place = "Greater Mumbai, Maharashtra, India"

print("Downloading Mumbai hospitals...")

mumbai_hospitals_raw = ox.features_from_place(
    place,
    {"amenity": "hospital"}
)

print("Downloaded hospitals:", len(mumbai_hospitals_raw))

Downloaded hospitals: 1780


In [10]:
mumbai_hospitals = mumbai_hospitals_raw.reset_index()

# Convert all geometries to representative points
mumbai_hospitals["geometry"] = (
    mumbai_hospitals["geometry"].representative_point()
)

mumbai_hospitals["latitude"] = mumbai_hospitals.geometry.y
mumbai_hospitals["longitude"] = mumbai_hospitals.geometry.x

if "name" not in mumbai_hospitals.columns:
    mumbai_hospitals["name"] = None

mumbai_hospitals = mumbai_hospitals[
    ["name", "latitude", "longitude"]
].copy()

mumbai_hospitals.to_csv(
    "../data/processed/mumbai_hospitals.csv",
    index=False
)

print("✅ Mumbai hospitals saved")
print("Rows:", len(mumbai_hospitals))
print(mumbai_hospitals.head())

✅ Mumbai hospitals saved
Rows: 1780
                                                name   latitude  longitude
0                              Saint Anne's Hospital  19.273380  72.883476
1  National Centre for Training and Research into...  19.051643  72.832132
2                    Jasmine Hospital And Wow Clinic  19.148671  72.999819
3                     Dhanvantari Hospitals Pvt.Ltd.  19.164547  73.239337
4                                      Borivali West  19.229813  72.847138


In [11]:
print("Downloading Mumbai schools...")

mumbai_schools_raw = ox.features_from_place(
    place,
    {"amenity": "school"}
)

print("Downloaded schools:", len(mumbai_schools_raw))

Downloaded schools: 798


In [12]:
mumbai_schools = mumbai_schools_raw.reset_index()

mumbai_schools["geometry"] = (
    mumbai_schools["geometry"].representative_point()
)

mumbai_schools["latitude"] = mumbai_schools.geometry.y
mumbai_schools["longitude"] = mumbai_schools.geometry.x

if "name" not in mumbai_schools.columns:
    mumbai_schools["name"] = None

mumbai_schools = mumbai_schools[
    ["name", "latitude", "longitude"]
].copy()

mumbai_schools.to_csv(
    "../data/processed/mumbai_schools.csv",
    index=False
)

print("✅ Mumbai schools saved")
print("Rows:", len(mumbai_schools))

✅ Mumbai schools saved
Rows: 798


In [13]:
print("Downloading Mumbai malls...")

mumbai_malls_raw = ox.features_from_place(
    place,
    {"shop": "mall"}
)

print("Downloaded malls:", len(mumbai_malls_raw))

Downloaded malls: 98


In [14]:
mumbai_malls = mumbai_malls_raw.reset_index()

mumbai_malls["geometry"] = (
    mumbai_malls["geometry"].representative_point()
)

mumbai_malls["latitude"] = mumbai_malls.geometry.y
mumbai_malls["longitude"] = mumbai_malls.geometry.x

if "name" not in mumbai_malls.columns:
    mumbai_malls["name"] = None

mumbai_malls = mumbai_malls[
    ["name", "latitude", "longitude"]
].copy()

mumbai_malls.to_csv(
    "../data/processed/mumbai_malls.csv",
    index=False
)

print("✅ Mumbai malls saved")
print("Rows:", len(mumbai_malls))

✅ Mumbai malls saved
Rows: 98


In [16]:
print("Downloading Mumbai metro/rail stations...")

mumbai_metro_raw = ox.features_from_place(
    place,
    {"railway": "station"}
)

print("Downloaded metro/rail stations:", len(mumbai_metro_raw))

Downloaded metro/rail stations: 265


In [17]:
mumbai_metro = mumbai_metro_raw.reset_index()

mumbai_metro["geometry"] = (
    mumbai_metro["geometry"].representative_point()
)

mumbai_metro["latitude"] = mumbai_metro.geometry.y
mumbai_metro["longitude"] = mumbai_metro.geometry.x

if "name" not in mumbai_metro.columns:
    mumbai_metro["name"] = None

mumbai_metro = mumbai_metro[
    ["name", "latitude", "longitude"]
].copy()

mumbai_metro.to_csv(
    "../data/processed/mumbai_metro_stations.csv",
    index=False
)

print("✅ Mumbai metro/rail stations saved")
print("Rows:", len(mumbai_metro))

✅ Mumbai metro/rail stations saved
Rows: 265


In [18]:
import osmnx as ox

place = "Delhi, India"

gdf = ox.geocode_to_gdf(place)

print(gdf[['display_name']])
print(gdf.geometry.geom_type)

   display_name
0  Delhi, India
0    Polygon
dtype: str


In [19]:
print("Downloading Delhi hospitals...")

delhi_hospitals_raw = ox.features_from_place(
    place,
    {"amenity": "hospital"}
)

print("Downloaded hospitals:", len(delhi_hospitals_raw))

Downloaded hospitals: 657


In [20]:
delhi_hospitals = delhi_hospitals_raw.reset_index()

delhi_hospitals["geometry"] = (
    delhi_hospitals["geometry"].representative_point()
)

delhi_hospitals["latitude"] = delhi_hospitals.geometry.y
delhi_hospitals["longitude"] = delhi_hospitals.geometry.x

if "name" not in delhi_hospitals.columns:
    delhi_hospitals["name"] = None

delhi_hospitals = delhi_hospitals[
    ["name", "latitude", "longitude"]
].copy()

delhi_hospitals.to_csv(
    "../data/processed/delhi_hospitals.csv",
    index=False
)

print("✅ Delhi hospitals saved")
print("Rows:", len(delhi_hospitals))

✅ Delhi hospitals saved
Rows: 657


In [21]:
print("Downloading Delhi schools...")

delhi_schools_raw = ox.features_from_place(
    place,
    {"amenity": "school"}
)

print("Downloaded schools:", len(delhi_schools_raw))

Downloaded schools: 1102


In [22]:
delhi_schools = delhi_schools_raw.reset_index()

delhi_schools["geometry"] = (
    delhi_schools["geometry"].representative_point()
)

delhi_schools["latitude"] = delhi_schools.geometry.y
delhi_schools["longitude"] = delhi_schools.geometry.x

if "name" not in delhi_schools.columns:
    delhi_schools["name"] = None

delhi_schools = delhi_schools[
    ["name", "latitude", "longitude"]
].copy()

delhi_schools.to_csv(
    "../data/processed/delhi_schools.csv",
    index=False
)

print("✅ Delhi schools saved")
print("Rows:", len(delhi_schools))

✅ Delhi schools saved
Rows: 1102


In [23]:
print("Downloading Delhi malls...")

delhi_malls_raw = ox.features_from_place(
    place,
    {"shop": "mall"}
)

print("Downloaded malls:", len(delhi_malls_raw))

Downloaded malls: 65


In [24]:
delhi_malls = delhi_malls_raw.reset_index()

delhi_malls["geometry"] = (
    delhi_malls["geometry"].representative_point()
)

delhi_malls["latitude"] = delhi_malls.geometry.y
delhi_malls["longitude"] = delhi_malls.geometry.x

if "name" not in delhi_malls.columns:
    delhi_malls["name"] = None

delhi_malls = delhi_malls[
    ["name", "latitude", "longitude"]
].copy()

delhi_malls.to_csv(
    "../data/processed/delhi_malls.csv",
    index=False
)

print("✅ Delhi malls saved")
print("Rows:", len(delhi_malls))

✅ Delhi malls saved
Rows: 65


In [25]:
print("Downloading Delhi metro/rail stations...")

delhi_metro_raw = ox.features_from_place(
    place,
    {"railway": "station"}
)

print("Downloaded metro/rail stations:", len(delhi_metro_raw))

Downloaded metro/rail stations: 269


In [26]:
delhi_metro = delhi_metro_raw.reset_index()

delhi_metro["geometry"] = (
    delhi_metro["geometry"].representative_point()
)

delhi_metro["latitude"] = delhi_metro.geometry.y
delhi_metro["longitude"] = delhi_metro.geometry.x

if "name" not in delhi_metro.columns:
    delhi_metro["name"] = None

delhi_metro = delhi_metro[
    ["name", "latitude", "longitude"]
].copy()

delhi_metro.to_csv(
    "../data/processed/delhi_metro_stations.csv",
    index=False
)

print("✅ Delhi stations saved")
print("Rows:", len(delhi_metro))

✅ Delhi stations saved
Rows: 269


In [27]:
place = "Pune, Maharashtra, India"

gdf = ox.geocode_to_gdf(place)

print(gdf[['display_name']])
print(gdf.geometry.geom_type)

                        display_name
0  Pune District, Maharashtra, India
0    Polygon
dtype: str


In [28]:
print("Downloading Pune hospitals...")

pune_hospitals_raw = ox.features_from_place(
    place,
    {"amenity": "hospital"}
)

print("Downloaded hospitals:", len(pune_hospitals_raw))

Downloaded hospitals: 980


In [29]:
pune_hospitals = pune_hospitals_raw.reset_index()

pune_hospitals["geometry"] = (
    pune_hospitals["geometry"].representative_point()
)

pune_hospitals["latitude"] = pune_hospitals.geometry.y
pune_hospitals["longitude"] = pune_hospitals.geometry.x

if "name" not in pune_hospitals.columns:
    pune_hospitals["name"] = None

pune_hospitals = pune_hospitals[
    ["name", "latitude", "longitude"]
].copy()

pune_hospitals.to_csv(
    "../data/processed/pune_hospitals.csv",
    index=False
)

print("✅ Pune hospitals saved")
print("Rows:", len(pune_hospitals))

✅ Pune hospitals saved
Rows: 980


In [30]:
print("Downloading Pune schools...")

pune_schools_raw = ox.features_from_place(
    place,
    {"amenity": "school"}
)

print("Downloaded schools:", len(pune_schools_raw))

ConnectTimeout: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x223c2a11450>, 'Connection to overpass-api.de timed out. (connect timeout=300)'))

In [31]:
ox.settings.overpass_url = "https://overpass.kumi.systems/api"
ox.settings.requests_timeout = 300

print("Using:", ox.settings.overpass_url)

Using: https://overpass.kumi.systems/api


In [33]:
pune_gdf = ox.geocode_to_gdf("Pune, Maharashtra, India")

bounds = pune_gdf.total_bounds
print("Bounds:", bounds)

Bounds: [73.3227109 17.8950735 75.1636505 19.3947459]


In [35]:
import pandas as pd

coordinates_final = pd.read_csv(
    "../data/processed/locality_coordinates_final.csv"
)

pune_coords = coordinates_final[
    (coordinates_final["city"] == "Pune") &
    (coordinates_final["latitude"].notna())
].copy()

print("Valid Pune localities:", len(pune_coords))

print("Latitude:",
      pune_coords["latitude"].min(),
      "→",
      pune_coords["latitude"].max())

print("Longitude:",
      pune_coords["longitude"].min(),
      "→",
      pune_coords["longitude"].max())

Valid Pune localities: 263
Latitude: 8.1833479 → 28.7518125
Longitude: 72.8211795 → 83.4291887


In [36]:
import osmnx as ox

place = "Pune Municipal Corporation, Maharashtra, India"

pune_city = ox.geocode_to_gdf(place)

print(pune_city[['display_name']])
print(pune_city.geometry.geom_type)
print("Bounds:", pune_city.total_bounds)

                                        display_name
0  Pune Municipal Corporation Madhavrao Sonba Tup...
0    Polygon
dtype: str
Bounds: [73.9384122 18.5058026 73.9391639 18.506551 ]


In [37]:
import osmnx as ox

place = {
    "city": "Pune",
    "state": "Maharashtra",
    "country": "India"
}

pune_city = ox.geocode_to_gdf(place)

print(pune_city[["display_name"]])
print(pune_city.geometry.geom_type)
print("Bounds:", pune_city.total_bounds)

TypeError: Nominatim did not geocode query {'city': 'Pune', 'state': 'Maharashtra', 'country': 'India'} to a geometry of type (Multi)Polygon.